# Условия

Приведены условия выполнения задания и логичный способ решения.

1. _**Дан датасет молекул для оценки.**_

   Все молекулы приведены в формате SDF в 2D представлении. Значит, можно использовать напрямую SMILES без необходимости учитывать стереохимию (ярко видно на примере 27 из датасета)

2. _**Нужно использовать `aizynthfinder` для работы**_

   Движка есть проблема - его сложно распараллелить из-за внутренней структуры. А так как нужно настроить именно его, будет проще скопировать структуры, и сделать параллельно работающий инструмент. (Для этих целей возможно лучше использовать `MultiAiZ`)

3. _**Найти числовую функцию синтетической сложности и вычислить её на датасете.**_

   Так как некоторые молекулы могут не иметь синтетического пути, функция должна учитывать этот факт и не обнулять значение в таком случае. Очевидно, нужно учитывать не только сложность пути и количество узлов, но и стоимость прекурсоров.

   Логично использовать уже готовые метрики, которые были провалидированы на реальных датасетах и имеют оценку экспертов.

4. _**Не указаны: база данных, стоимость доступных реагентов, предпочтительные реакции**_

   Без этой информации можно использовать стандартный набор реагентов с их стоимостью. Например, тот, который скачивается вместе с библиотекой `aizynthfinder` (модели фильтрации и поиска пути, обученные на `UPSTO` базе данных реакций, `ZINC` - база прекурсоров).

   Исходя из этого, можно напрямую использовать метрику `RAscore`, которая обучалась на этих же данных. Подойдёт и любая другая, которая использует `aizynthfinder` в качестве отправной точки.

5. _**Нужны данные о скорости расчёта.**_

   Молекул много, значит нужно использовать параллельные вычисления. Нужно логирование или хотя бы сохранение времени расчёта для каждой отдельной молекулы.


# Импорты

Обертка для ретросинтеза импортируется как набор модулей для отдельных задач:

- `config` - настройки движка
- `scorers` - набор доступных скорринговых функций
- `draw` - инструменты для отчётов, рисования и так далее.
- `retro` - основные функции ретросинтеза.


In [1]:
from __future__ import annotations

from rdkit import Chem as rd
from chemrar_retro import config, retro, scorers, draw
from pathlib import Path
import pandas as pd
import numpy as np

# Загрузка молекул датасета.


In [2]:
mols = retro.mols_from_sdf(Path('data/output_500.sdf'))

# Настройка движка и функций скоринга


In [ ]:
# Цена всего
scorers.price.RouteCostScorer

# Доступность в стоке
scorers.availability. FractionInStockScorer

In [ ]:
scorer = {

}

In [ ]:
engine = retro.create_engine(
    expansion=dict(
        uspto=config.ExpansionPolicy(
            model=Path("download/example/uspto_model.onnx"),
            template=Path("download/example/uspto_templates.csv.gz"),
        ),
        ringbreaker=config.ExpansionPolicy(
            model=Path("download/example/uspto_ringbreaker_model.onnx"),
            template=Path("download/example/uspto_ringbreaker_templates.csv.gz"),
        ),
    ),
    filter=dict(
        uspto=config.FilterPolicy(model=Path("download/example/uspto_filter_model.onnx")),
    ),
    stock=dict(
        zinc=config.Stock(path=Path("download/example/zinc_stock.hdf5")),
    ),
    scorers=[(scorers.availability. FractionInStockScorer, {})],
)


# Ретросинтез


In [ ]:
engine_selected = retro.select(
    engine,
    stocks=engine.stock.items,
    filter_policies=engine.filter_policy.items,
    search_scorers={"state score": 1},
    expansion_policies=engine.expansion_policy.items,
)

In [ ]:
tree = retro.generate_tree(engine_selected, smiles)
retro.search_tree(tree, 100)
result = retro.analyze_tree(tree, scorers=["fraction in stock", "state score"], top_n=10)